# Self Merge Rate - Visualization

### Importing Libraries

In [ ]:
import sys
import subprocess
import pandas as pd
import sqlalchemy as salc
import json
import os
from pathlib import Path
import datetime as dt
import plotly.graph_objects as go



## Installing Dependencies - Package Management

* This particular implementation uses requirements.txt to install the dependencies

In [ ]:
# Package management - Installing requirements.txt
_repo_root = Path.cwd().parent if Path.cwd().name == "wg-data-science" else Path.cwd()
_req_notebook = Path(__file__).parent / "requirements.txt" if "__file__" in dir() else _repo_root / "wg-data-science" / "requirements.txt"
if not _req_notebook.exists():
    _req_notebook = _repo_root / "requirements.txt"


if _req_notebook.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(_req_notebook)], check=True)
    print("Installed notebook deps from", _req_notebook)
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "pandas", "sqlalchemy", "plotly", "psycopg2-binary"],
        check=True,
    )
    print("Installed minimal deps (requirements.txt not found)")

### Loading Credentials - .env; .json...

In [ ]:
with open("../augur_creds.json") as config_file:
    config = json.load(config_file)

### DB connection

In [ ]:
database_connection_string = "postgresql+psycopg2://{}:{}@{}:{}/{}".format(
    config["user"], config["password"], config["host"], config["port"], config["database"]
)
dbschema = "augur_data"
engine = salc.create_engine(
    database_connection_string,
    connect_args={"options": "-csearch_path={}".format(dbschema)},
)

### Queries

In [ ]:
repo_urls = ["https://github.com/chaoss/augur"]

url_query = str(repo_urls)[1:-1]
repo_query = salc.sql.text(
    f"""
    SET SCHEMA 'augur_data';
    SELECT DISTINCT r.repo_id, r.repo_name
    FROM repo r
    JOIN repo_groups rg ON r.repo_group_id = rg.repo_group_id
    WHERE r.repo_git IN ({url_query})
    """
)
with engine.connect() as conn:
    t = conn.execute(repo_query)
    results = t.all()
repo_ids = [row[0] for row in results]
repo_names = [row[1] for row in results]
print(repo_ids)
print(repo_names)

In [ ]:
repo_statement = str(repo_ids)[1:-1]

query = salc.sql.text(f"""
    SELECT
        r.repo_id,
        r.repo_name,
        pr.pull_request_id,
        pr.pr_src_number,
        left(pr.pr_augur_contributor_id::text, 15) AS cntrb_id,
        pr.pr_created_at AS created_at,
        pr.pr_merged_at AS merged_at,
        left(pre.cntrb_id::text, 15) AS merger_cntrb_id
    FROM repo r
    JOIN pull_requests pr ON r.repo_id = pr.repo_id
    LEFT JOIN pull_request_events pre
        ON pr.pull_request_id = pre.pull_request_id AND pre.action = 'merged'
    WHERE r.repo_id IN ({repo_statement})
      AND pr.pr_merged_at IS NOT NULL
      AND pr.pr_merged_at < now()
    ORDER BY pr.pr_merged_at
""")
df = pd.read_sql(query, con=engine)
df = df.reset_index()
df.drop("index", axis=1, inplace=True)
df["merged_at"] = pd.to_datetime(df["merged_at"], utc=True)
df["created_at"] = pd.to_datetime(df["created_at"], utc=True)

### Data Processing

In [ ]:
df

In [ ]:
INTERVAL = "M"

In [ ]:
# Self-merge: author == merger (with safe handling of nulls)
df["cntrb_id"] = df["cntrb_id"].astype(str)
df["merger_cntrb_id"] = df["merger_cntrb_id"].fillna("").astype(str)
df["is_self_merge"] = (df["cntrb_id"] == df["merger_cntrb_id"]) & (df["merger_cntrb_id"] != "")

# Period slice for date formatting
period_slice = 10 if INTERVAL == "W" else (4 if INTERVAL == "Y" else 7)

# Counts per period by merged_at
merged_range = df["merged_at"].dt.to_period(INTERVAL).value_counts().sort_index()
df_merged = merged_range.to_frame().reset_index().rename(columns={"merged_at": "Date", "count": "total_merged"})
df_merged["Date"] = pd.to_datetime(df_merged["Date"].astype(str).str[:period_slice])

self_merge_range = df[df["is_self_merge"]]["merged_at"].dt.to_period(INTERVAL).value_counts().sort_index()
df_self = self_merge_range.to_frame().reset_index().rename(columns={"merged_at": "Date", "count": "self_merged"})
df_self["Date"] = pd.to_datetime(df_self["Date"].astype(str).str[:period_slice])

df_plot = df_merged.merge(df_self, on="Date", how="outer").fillna(0)
df_plot["self_merged"] = df_plot["self_merged"].astype(int)
df_plot["rate_pct"] = (df_plot["self_merged"] / df_plot["total_merged"].replace(0, float("nan")) * 100).round(1)


if INTERVAL == "M":
    df_plot["Date_str"] = pd.to_datetime(df_plot["Date"]).dt.strftime("%Y-%m-01")
elif INTERVAL == "Y":
    df_plot["Date_str"] = pd.to_datetime(df_plot["Date"]).dt.strftime("%Y-01-01")
else:
    df_plot["Date_str"] = pd.to_datetime(df_plot["Date"]).dt.strftime("%Y-%m-%d")

df_plot = df_plot.sort_values("Date").reset_index(drop=True)
df_plot

### Plots

In [ ]:
fig = go.Figure(
    [
        go.Scatter(
            name="Merged (total)",
            x=df_plot["Date_str"],
            y=df_plot["total_merged"],
            mode="lines",
            showlegend=True,
            hovertemplate="Merged (total): %{y}<br>%{x|%b %d, %Y} <extra></extra>",
        ),
        go.Scatter(
            name="Self-merged",
            x=df_plot["Date_str"],
            y=df_plot["self_merged"],
            mode="lines",
            showlegend=True,
            hovertemplate="Self-merged: %{y}<br>%{x|%b %d, %Y} <extra></extra>",
        ),
    ]
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Number of PRs",
    font=dict(size=14),
    title="Self Merge Rates",
)

In [ ]:
fig_rate = go.Figure()
fig_rate.add_trace(
    go.Scatter(
        x=df_plot["Date_str"],
        y=df_plot["rate_pct"],
        mode="lines",
        name="Self-merge rate (%)",
        marker=dict(size=6),
        hovertemplate="%{x}<br>Rate: %{y:.1f}%<extra></extra>",
    )
)
fig_rate.update_layout(
    title="Self-merge rate (%) over time",
    xaxis_title="Year",
    yaxis_title="Self-merge rate (%)",
    font=dict(size=14),
)
fig_rate.show()